In [1]:
!pip install pandas numpy matplotlib scikit-learn tqdm --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Problems to solve

1. Download the data from the [Don'tGetKicked contest](https://www.kaggle.com/c/DontGetKicked).

    Design a train/validation/test split.

    Use the "PurchDate" field for the split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.
    Use the first 33% of the dates for the train, the last 33% of the dates for the test, and the middle 33% for the validation set. *Don't use the test dataset until the end!*

    Use sklearn's LabelEncoder or OneHotEncoder to preprocess categorical variables. Be careful with data leakage (fit Encoder to train and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training). [Example](https://contrib.scikit-learn.org/category_encoders/count.html)
2. Create a Python class, MLP with 1 hidden layer and sigmoid activation function at the end of the network.
It should support *fit, predict_proba* and *predict methods*. Also, the number of neurons in the hidden layer and the activation function must be parameters of your class.

    Here is the blueprint:

    ```python
    model = MLP(n_hidden=100, activation=np.arctan)
    model.fit(Xtrain, ytrain)
    model.predict_proba(Xvalid)
    ```
    <br>
    
    - Initialize the network with small random numbers.
    - Use Log Loss (Binary Cross Entropy) as the loss function.
    - Implement a forward pass; you can use a fixed batch size like 32, forward pass maps arrays of shape (batch_size, number_of_features) to arrays of shape (batch_size, 2) where 2 means dimensions of [probability_of_0, probability_of_1] output space.
    - Write down the gradients of the loss function with respect to the parameters of the net. Use [4](https://en.wikipedia.org/wiki/Backpropagation), [9](http://d2l.ai/) as a guide for deriving gradients.
    - Use gradients to perform Backprop. Implement Backprop.
    - Implement SGD algorithm to tune model parameters.
    - Design a basic train-validation loop: iterate over the training dataset, batch by batch, update the parameters of the network, and check the quality of the model using the validation set.
    - Write code to update network weights using SGD or Adam.<br><br>
3. With your MLP module and careful network engineering, you must obtain at least a 0.15 Gini score on the validation dataset. You can train for more than 1 epoch, use different activation functions, and different optimizers (like SGD or Adam).
4. Use sklearn's MLPClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?
5. Implement and try different activation functions: sigmoid, ReLU, cosine. Remember that you have to derive gradients for each different activation function. Which activation function gives the best performance on the validation dataset?
6. Design an MLP module with 1 hidden layer using any high level deep learning framework: Pytorch, Keras, or Tensorflow. Check its performance on the validation dataset.  Is it better than your module? If so, why?
7. Take the best model and estimate its performance on the test dataset: check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitting or not? Explain.
8. **Bonus Part**: Implement the Adam algorithm to tune the parameters of the model.

### Import libraries

In [2]:
import torch.utils.data as data
import torch.optim as optim
import torch.nn as nn
import torch

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

import pandas.api.types as ptypes
import pandas as pd
import numpy as np

# 1.

In [3]:
raw_df = pd.read_csv("./data/training.zip", header=0, parse_dates=["PurchDate"])

**Useless features for predicting**: `PurchDate`, `Color`, `AUCGUART` (too many NaNs), `RefId`, `PRIMEUNIT` (too many NaNs), `BYRNO` (it's an ID column), `TopThreeAmericanName` (dublicates 'Make' column), `SubModel` (combines `Model` + `Trim`)<br><br>
**Under investigation**:  `Auction` (>55% - MANHEIN, >23% - OTHER), `Nationality`(~85% - AMERICAN)<br><br>
**Geo features**: `VNZIP1`, `VNST`

### Cleaning data

In [4]:
raw_df.drop(["Color", "AUCGUART", "RefId", "PRIMEUNIT", "BYRNO", "TopThreeAmericanName", "SubModel"], axis='columns', inplace=True)

### Sorting by `PurchDate`

In [5]:
raw_df.set_index("PurchDate", drop=True, inplace=True)
raw_df.sort_index(axis="index", inplace=True, ignore_index=True)

raw_df

,IsBadBuy,Auction,VehYear,VehicleAge,Make,Model,Trim,Transmission,WheelTypeID,WheelType,...,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,0,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,AUTO,2.0,Covers,...,10066.0,8709.0,10331.0,9906.0,11657.0,80022,CO,6770.0,0,1389
1,0,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,AUTO,1.0,Alloy,...,6693.0,4908.0,5971.0,5801.0,6949.0,80022,CO,6160.0,0,941
2,0,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,AUTO,2.0,Covers,...,4886.0,3397.0,4272.0,4169.0,5114.0,80022,CO,4250.0,0,1155
3,0,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,AUTO,1.0,Alloy,...,11174.0,9202.0,10794.0,10438.0,12158.0,80022,CO,8180.0,0,1703
4,0,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,AUTO,1.0,Alloy,...,5068.0,3369.0,4492.0,4139.0,5351.0,80022,CO,4900.0,0,825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,0,ADESA,2006,4,SUZUKI,AERIO,SX,AUTO,1.0,Alloy,...,9267.0,4411.0,4872.0,7601.0,8041.0,28273,NC,4500.0,0,983
72979,1,ADESA,2006,4,SUZUKI,GRAND VITARA 2WD,NaN,AUTO,NaN,NaN,...,11604.0,7050.0,8061.0,11114.0,11830.0,28273,NC,7470.0,0,920
72980,1,ADESA,2005,5,FORD,TAURUS,SE,AUTO,1.0,Alloy,...,7203.0,3364.0,4678.0,6476.0,9005.0,28273,NC,4130.0,0,1053
72981,1,MANHEIM,2005,5,CHEVROLET,MALIBU 4C,Cla,AUTO,2.0,Covers,...,8588.0,3643.0,5100.0,6908.0,8446.0,92337,CA,4395.0,0,1243


### Filling NA-values

In [6]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   IsBadBuy                           72983 non-null  int64  
 1   Auction                            72983 non-null  str    
 2   VehYear                            72983 non-null  int64  
 3   VehicleAge                         72983 non-null  int64  
 4   Make                               72983 non-null  str    
 5   Model                              72983 non-null  str    
 6   Trim                               70623 non-null  str    
 7   Transmission                       72974 non-null  str    
 8   WheelTypeID                        69814 non-null  float64
 9   WheelType                          69809 non-null  str    
 10  VehOdo                             72983 non-null  int64  
 11  Nationality                        72978 non-null  str    
 12  S

In [7]:
raw_df.value_counts(["WheelType", "WheelTypeID"])

WheelType  WheelTypeID
Alloy      1.0            36050
Covers     2.0            33004
Special    3.0              755
Name: count, dtype: int64

<br><br>Thus, one column is **excess**. `WheelTypeID` adds relations `>` or `<` which have no sence for the WheelType. So this column should be removed.<br><br>

In [8]:
raw_df.drop(["WheelTypeID"], axis="columns", inplace=True)

<br><br>First of all, before *filling NA-values* with mean and mode, column `Trim` requires filling, because it will be used as *an index* when aggregating.<br><br>In addition, all columns containing string need transforming by means of `str.lower`<br><br>

In [9]:
for column in raw_df:
    if ptypes.is_string_dtype(raw_df[column]):
        raw_df[column] = raw_df[column].str.lower()

In [10]:
class Filler:
    def __init__(self, handle_int: bool = True):
        self.handle_int = handle_int

    def __call__(self, df: pd.DataFrame, cols_agg: Iterable[str], *args, **kwargs) -> None:
        if not isinstance(df, pd.DataFrame):
            raise TypeError("'df' must be a pd.DataFrame")

        cols_agg = list(cols_agg)

        if any(not isinstance(elem, str) for elem in cols_agg):
            raise TypeError("Each element of 'cols_agg' must be a string")

        if any(not elem in df for elem in cols_agg):
            raise ValueError("'df' must contain the columns named with the strings from 'cols_agg'")

        for column in df.columns:
            if column in cols_agg:
                continue
            if df[column].isna().any():
                func = self.fill_with_mode if self.check_type(df, column) else self.fill_with_mean
                df[column] = df.groupby(cols_agg)[column].transform(func)

    def check_type(self, df: pd.DataFrame, column: str) -> bool:
        flag_1 = isinstance(df[column].dtype, (pd.StringDtype, pd.CategoricalDtype))
        flag_2 = self.handle_int and ptypes.is_integer_dtype(df[column])

        return flag_1 or flag_2
    
    @staticmethod
    def fill_with_mode(group):
        mode = group.mode()
        if not mode.empty:
            fill_val = mode.iloc[0]
            return group.fillna(value=fill_val)
        else:
            return group

    @staticmethod
    def fill_with_mean(group):
        mean = group.mean()
        if pd.notna(mean):
            return group.fillna(value=mean)
        else:
            return group

    @property
    def handle_int(self):
        return self._handle_int

    @handle_int.setter
    def handle_int(self, handle_int):
        if not isinstance(handle_int, bool):
            raise TypeError("Parameter 'handle_int' must be boolean")

        self._handle_int = handle_int

In [11]:
filler = Filler()

raw_df["Trim"] = raw_df.groupby(["Make", "Model"])["Trim"].transform(filler.fill_with_mode)
raw_df["Trim"] = raw_df["Trim"].fillna(value="unknown")

filler(raw_df, ["Make", "Model", "Trim"])
filler(raw_df, ["Make", "Model"])
filler(raw_df, ["Make"])

raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   IsBadBuy                           72983 non-null  int64  
 1   Auction                            72983 non-null  str    
 2   VehYear                            72983 non-null  int64  
 3   VehicleAge                         72983 non-null  int64  
 4   Make                               72983 non-null  str    
 5   Model                              72983 non-null  str    
 6   Trim                               72983 non-null  str    
 7   Transmission                       72983 non-null  str    
 8   WheelType                          72983 non-null  str    
 9   VehOdo                             72983 non-null  int64  
 10  Nationality                        72983 non-null  str    
 11  Size                               72983 non-null  str    
 12  M

### Splitting train/valid/test

In [12]:
length = len(raw_df)
third = length // 3

X_train = raw_df.iloc[:third, 1:].copy()
X_train.reset_index(drop=True, inplace=True)

X_valid = raw_df.iloc[third:(2 * third), 1:].copy()
X_valid.reset_index(drop=True, inplace=True)

X_test = raw_df.iloc[(2 * third):, 1:].copy()
X_test.reset_index(drop=True, inplace=True)

y_train = raw_df.iloc[:third, 0].copy()
y_train.reset_index(drop=True, inplace=True)

y_valid = raw_df.iloc[third:(2 * third), 0].copy()
y_valid.reset_index(drop=True, inplace=True)

y_test = raw_df.iloc[(2 * third):, 0].copy()
y_test.reset_index(drop=True, inplace=True)

del raw_df

### OneHotEncoding & Scaling

In [13]:
cat_feat = list()

for column in X_train.columns:
    if ptypes.is_string_dtype(X_train[column]):
        cat_feat.append(column)

print(*cat_feat, sep="  ")

Auction  Make  Model  Trim  Transmission  WheelType  Nationality  Size  VNST


In [14]:
encoder = OneHotEncoder(dtype=np.int8, handle_unknown="ignore", min_frequency=24)
scaler = StandardScaler()

encoder.fit(X_train.loc[:, cat_feat])

train_binary_val = encoder.transform(X_train.loc[:, cat_feat])
valid_binary_val = encoder.transform(X_valid.loc[:, cat_feat])
test_binary_val = encoder.transform(X_test.loc[:, cat_feat])

train_binary_feat = pd.DataFrame(train_binary_val.toarray(), columns=encoder.get_feature_names_out())
valid_binary_feat = pd.DataFrame(valid_binary_val.toarray(), columns=encoder.get_feature_names_out())
test_binary_feat = pd.DataFrame(test_binary_val.toarray(), columns=encoder.get_feature_names_out())

X_train.drop(cat_feat, axis="columns", inplace=True)
X_valid.drop(cat_feat, axis="columns", inplace=True)
X_test.drop(cat_feat, axis="columns", inplace=True)

scaler.fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), columns=scaler.feature_names_in_)
X_valid = pd.DataFrame(scaler.transform(X_valid), columns=scaler.feature_names_in_)
X_test = pd.DataFrame(scaler.transform(X_test), columns=scaler.feature_names_in_)

X_train = pd.concat([X_train, train_binary_feat], axis="columns")
X_valid = pd.concat([X_valid, valid_binary_feat], axis="columns")
X_test = pd.concat([X_test, test_binary_feat], axis="columns")

In [15]:
X_train.head(7)

,VehYear,VehicleAge,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,...,VNST_nv,VNST_oh,VNST_ok,VNST_pa,VNST_sc,VNST_tn,VNST_tx,VNST_ut,VNST_va,VNST_infrequent_sklearn
0,1.357265,-1.357240,0.608776,0.586899,0.656712,0.585774,0.655103,1.149197,1.165902,1.057817,...,0,0,0,0,0,0,0,0,0,0
1,0.109884,-0.109905,-2.251226,-0.556675,-0.464981,-0.550379,-0.459670,-0.373579,-0.399654,-0.404534,...,0,0,0,0,0,0,0,0,0,0
2,-0.513807,0.513762,0.128598,-1.082350,-1.065875,-1.072715,-1.056882,-0.978924,-1.009719,-0.985912,...,0,0,0,0,0,0,0,0,0,0
3,0.733574,-0.733573,-0.027052,1.095733,1.025222,1.091142,1.021296,1.346705,1.332153,1.247335,...,0,0,0,0,0,0,0,0,0,0
4,-0.513807,0.513762,-0.231833,-1.083954,-1.005175,-1.074191,-0.996731,-0.990141,-0.930723,-0.996599,...,0,0,0,0,0,0,0,0,0,0
5,0.109884,-0.109905,0.790601,-0.622836,-0.588895,-0.616040,-0.582946,-0.449297,-0.440589,-0.477206,...,0,0,0,0,0,0,0,0,0,0
6,-1.137498,1.137430,-0.429755,-0.824124,-0.806194,-0.816343,-0.798762,-0.309479,-0.362311,-0.343261,...,0,0,0,0,0,0,0,0,0,0


# 2 - 3.

In [16]:
def gini_score(y: np.ndarray, p: np.ndarray) -> float:
    return (2 * roc_auc_score(y, p) - 1)

In [17]:
%run ./modules/optimizers.py
%run ./modules/mlp.py

Module containing SGD/Adam optimizers imported✅
Module containing MLP classifier imported✅


In [18]:
n_hidden = 32
batch_size, n_epoch = 32, 20
optim_class, optim_params = Adam, dict(lr=0.001)

model = MLP(n_hidden=n_hidden, activation=np.tanh)
model.fit(X_train, y_train, batch_size=batch_size, n_epoch=n_epoch, optim_class=optim_class, optim_params=optim_params)

Epoch [1/20], loss_mean=0.356: : 761it [00:00, 2387.92it/s]
Epoch [2/20], loss_mean=0.333: : 761it [00:00, 2580.69it/s]
Epoch [3/20], loss_mean=0.330: : 761it [00:00, 2501.21it/s]
Epoch [4/20], loss_mean=0.330: : 761it [00:00, 2170.92it/s]
Epoch [5/20], loss_mean=0.329: : 761it [00:00, 2377.71it/s]
Epoch [6/20], loss_mean=0.329: : 761it [00:00, 2413.13it/s]
Epoch [7/20], loss_mean=0.328: : 761it [00:00, 1646.07it/s]
Epoch [8/20], loss_mean=0.328: : 761it [00:00, 2040.59it/s]
Epoch [9/20], loss_mean=0.327: : 761it [00:00, 2480.80it/s]
Epoch [10/20], loss_mean=0.327: : 761it [00:00, 1456.19it/s]
Epoch [11/20], loss_mean=0.327: : 761it [00:00, 2558.98it/s]
Epoch [12/20], loss_mean=0.326: : 761it [00:00, 2150.60it/s]
Epoch [13/20], loss_mean=0.326: : 761it [00:00, 1658.35it/s]
Epoch [14/20], loss_mean=0.326: : 761it [00:00, 1948.57it/s]
Epoch [15/20], loss_mean=0.325: : 761it [00:00, 2416.55it/s]
Epoch [16/20], loss_mean=0.325: : 761it [00:00, 1544.03it/s]
Epoch [17/20], loss_mean=0.324: :

In [19]:
probas = model.predict_proba(X_valid)
print(f"\nFinal gini_score for valid: {gini_score(y_valid, probas[:, 1]):.03f}")


Final gini_score for valid: 0.300


# 4.

In [20]:
model = MLPClassifier(hidden_layer_sizes=(32,), activation='tanh', solver='adam', batch_size=batch_size, learning_rate_init=0.001, max_iter=1000, random_state=21)
model.fit(X_train, y_train)

probas = model.predict_proba(X_valid)
print(f"\nFinal gini_score for valid: {gini_score(y_valid, probas[:, 1]):.03f}")


Final gini_score for valid: 0.182


<br>As far as Gini score is concerned, my implementation **exceeds** the sklearn's one **by far**. A *huger range of hyperparameters* to configure is the main reason for this behavior followed by a *different weights/biases initialization technique*.

# 5.

In [21]:
### Sigmoid
sigm = lambda x: 1 / (1 + np.exp(-x))

n_hidden = 32
batch_size, n_epoch = 32, 20
optim_class, optim_params = Adam, dict(lr=0.001)

model = MLP(n_hidden=n_hidden, activation=sigm)
model.fit(X_train, y_train, batch_size=batch_size, n_epoch=n_epoch, optim_class=optim_class, optim_params=optim_params)

Epoch [1/20], loss_mean=0.356: : 761it [00:00, 2122.87it/s]
Epoch [2/20], loss_mean=0.335: : 761it [00:00, 2164.43it/s]
Epoch [3/20], loss_mean=0.333: : 761it [00:00, 2534.59it/s]
Epoch [4/20], loss_mean=0.331: : 761it [00:00, 1816.16it/s]
Epoch [5/20], loss_mean=0.331: : 761it [00:00, 2195.45it/s]
Epoch [6/20], loss_mean=0.330: : 761it [00:00, 1995.99it/s]
Epoch [7/20], loss_mean=0.329: : 761it [00:00, 1589.86it/s]
Epoch [8/20], loss_mean=0.329: : 761it [00:00, 2367.87it/s]
Epoch [9/20], loss_mean=0.328: : 761it [00:00, 2197.32it/s]
Epoch [10/20], loss_mean=0.328: : 761it [00:00, 1565.22it/s]
Epoch [11/20], loss_mean=0.328: : 761it [00:00, 2081.29it/s]
Epoch [12/20], loss_mean=0.327: : 761it [00:00, 2110.48it/s]
Epoch [13/20], loss_mean=0.327: : 761it [00:00, 1577.25it/s]
Epoch [14/20], loss_mean=0.328: : 761it [00:00, 1923.82it/s]
Epoch [15/20], loss_mean=0.327: : 761it [00:00, 2214.62it/s]
Epoch [16/20], loss_mean=0.327: : 761it [00:00, 1391.67it/s]
Epoch [17/20], loss_mean=0.327: :

In [22]:
probas = model.predict_proba(X_valid)
print(f"\nFinal gini_score for valid: {gini_score(y_valid, probas[:, 1]):.03f}")


Final gini_score for valid: 0.314


In [23]:
### ReLU
relu = lambda x: (x >= 0).astype(int) * x

n_hidden = 32
batch_size, n_epoch = 32, 20
optim_class, optim_params = Adam, dict(lr=0.001)

model = MLP(n_hidden=n_hidden, activation=relu)
model.fit(X_train, y_train, batch_size=batch_size, n_epoch=n_epoch, optim_class=optim_class, optim_params=optim_params)

Epoch [1/20], loss_mean=0.358: : 761it [00:00, 2074.98it/s]
Epoch [2/20], loss_mean=0.332: : 761it [00:00, 2301.52it/s]
Epoch [3/20], loss_mean=0.330: : 761it [00:00, 2414.81it/s]
Epoch [4/20], loss_mean=0.329: : 761it [00:00, 1689.83it/s]
Epoch [5/20], loss_mean=0.328: : 761it [00:00, 1958.85it/s]
Epoch [6/20], loss_mean=0.328: : 761it [00:00, 2210.72it/s]
Epoch [7/20], loss_mean=0.327: : 761it [00:00, 1628.90it/s]
Epoch [8/20], loss_mean=0.327: : 761it [00:00, 1963.03it/s]
Epoch [9/20], loss_mean=0.326: : 761it [00:00, 2277.30it/s]
Epoch [10/20], loss_mean=0.325: : 761it [00:00, 1530.07it/s]
Epoch [11/20], loss_mean=0.325: : 761it [00:00, 2011.44it/s]
Epoch [12/20], loss_mean=0.324: : 761it [00:00, 2422.35it/s]
Epoch [13/20], loss_mean=0.324: : 761it [00:00, 1528.81it/s]
Epoch [14/20], loss_mean=0.324: : 761it [00:00, 2417.85it/s]
Epoch [15/20], loss_mean=0.323: : 761it [00:00, 2228.52it/s]
Epoch [16/20], loss_mean=0.322: : 761it [00:00, 1664.01it/s]
Epoch [17/20], loss_mean=0.322: :

In [24]:
probas = model.predict_proba(X_valid)
print(f"\n\nFinal gini_score for valid: {gini_score(y_valid, probas[:, 1]):.03f}")



Final gini_score for valid: 0.295


In [25]:
### Cosine
n_hidden = 32
batch_size, n_epoch = 32, 20
optim_class, optim_params = Adam, dict(lr=0.001)

model = MLP(n_hidden=n_hidden, activation=np.cos)
model.fit(X_train, y_train, batch_size=batch_size, n_epoch=n_epoch, optim_class=optim_class, optim_params=optim_params)

Epoch [1/20], loss_mean=0.352: : 761it [00:00, 2268.13it/s]
Epoch [2/20], loss_mean=0.334: : 761it [00:00, 2339.11it/s]
Epoch [3/20], loss_mean=0.331: : 761it [00:00, 2342.22it/s]
Epoch [4/20], loss_mean=0.330: : 761it [00:00, 1657.63it/s]
Epoch [5/20], loss_mean=0.329: : 761it [00:00, 1825.99it/s]
Epoch [6/20], loss_mean=0.329: : 761it [00:00, 2523.41it/s]
Epoch [7/20], loss_mean=0.328: : 761it [00:00, 1649.01it/s]
Epoch [8/20], loss_mean=0.328: : 761it [00:00, 1784.47it/s]
Epoch [9/20], loss_mean=0.327: : 761it [00:00, 2353.52it/s]
Epoch [10/20], loss_mean=0.326: : 761it [00:00, 1617.97it/s]
Epoch [11/20], loss_mean=0.326: : 761it [00:00, 2118.58it/s]
Epoch [12/20], loss_mean=0.325: : 761it [00:00, 2262.78it/s]
Epoch [13/20], loss_mean=0.325: : 761it [00:00, 1458.52it/s]
Epoch [14/20], loss_mean=0.325: : 761it [00:00, 1963.03it/s]
Epoch [15/20], loss_mean=0.324: : 761it [00:00, 2262.66it/s]
Epoch [16/20], loss_mean=0.323: : 761it [00:00, 1468.23it/s]
Epoch [17/20], loss_mean=0.322: :

In [26]:
probas = model.predict_proba(X_valid)
print(f"\n\nFinal gini_score for valid: {gini_score(y_valid, probas[:, 1]):.03f}")



Final gini_score for valid: 0.291


<br>As far as the Gini score on valdation dataset is concerned, *sigmoid activation function* exceeds others.

# 6.

In [27]:
class AuctionDataset(data.Dataset):
    def __init__(self, x: pd.DataFrame | np.ndarray, y: pd.Series | np.ndarray):
        if not isinstance(x, (pd.DataFrame, np.ndarray)):
            raise TypeError(f"'x' must be either pandas.DataFrame or numpy.ndarray")
        if not isinstance(y, (pd.Series, np.ndarray)):
            raise TypeError(f"'y' must be either pandas.Series or numpy.ndarray")

        x = x.values if isinstance(x, pd.DataFrame) else x
        x = x.astype(np.float32)
        
        y = y.values if isinstance(y, pd.Series) else y
        y = y.astype(np.float32)

        self.data = torch.from_numpy(x)
        self.target = torch.from_numpy(y)

    def __getitem__(self, item: int):
        return self.data[item, :], self.target[item]

    def __len__(self):
        return self.data.shape[0]

In [28]:
class OneLayerNetwork(nn.Module):
    def __init__(self, input_dim: int, hidd_dim: int, output_dim: int):
        super().__init__()
        
        self.lin_layer_1 = nn.Linear(input_dim, hidd_dim, bias=True)
        self.act_fc_1 = nn.Sigmoid()
        self.lin_layer_2 = nn.Linear(hidd_dim, output_dim, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self.lin_layer_1(x)
        output = self.act_fc_1(output)
        output = self.lin_layer_2(output)

        return output

In [29]:
train_ds, valid_ds, test_ds = AuctionDataset(X_train, y_train), AuctionDataset(X_valid, y_valid), AuctionDataset(X_test, y_test)
train_loader = data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_loader = data.DataLoader(valid_ds, batch_size=len(valid_ds), shuffle=False)
test_loader = data.DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

model = OneLayerNetwork(X_train.shape[1], n_hidden, 1)
model.train()

optimizer = optim.Adam(params=model.parameters())
loss_func = nn.BCEWithLogitsLoss()

for epoch in range(n_epoch):
    for batch, target in train_loader:
        p = model(batch).squeeze_()
        loss = loss_func(p, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [30]:
model.eval()
with torch.no_grad():
    for batch, target in valid_loader:
        logits = model(batch).squeeze_()
        probas = torch.sigmoid(logits)
        probas = probas.numpy()
        print(f"\n\nFinal gini_score for valid: {gini_score(y_valid, probas):.03f}")



Final gini_score for valid: 0.308


<br>In terms of the Gini score on validation dataset PyTorch implementation showcased *a slightly higher value*. The main reasons of it are: 
- a huger range of hyperparameters;
- a smart weights/biases initialization technique;
- more precise values for gradients/derivatives;
- higher computational accuracy.

# 7.

In [31]:
model.eval()
with torch.no_grad():
    for batch, target in test_loader:
        logits = model(batch).squeeze_()
        probas = torch.sigmoid(logits)
        probas = probas.numpy()
        print(f"\n\nFinal gini_score for test: {gini_score(y_test, probas):.03f}")



Final gini_score for test: 0.282


In [32]:
with torch.no_grad():
    all_probas = []
    
    for batch, target in train_loader:
        logits = model(batch).squeeze_()
        probas = torch.sigmoid(logits)
        probas = probas.numpy()
        all_probas.append(probas)
        
    probas = np.concatenate(all_probas)
    print(f"\nFinal gini_score for train: {gini_score(y_train, probas):.03f}")


Final gini_score for train: 0.006


For the validation part Gini score is slightly higher (~$0.027$) indicating the absence of overfitting (the difference < $10$%). An interesting thing is that the score for training dataset is much lower (it literally means that model quesses: $50$/$50$). So the final model is not overfitted but it's obviously **underfitted**. It's not a surprice because the class only based on one hidden layer.